# E46 — TensorRT on a real T4 (Colab dry-run for DACON)

Validates the DACON TensorRT submission flow on **actual T4 hardware** (free Colab tier):
1. time `pip install tensorrt` (DACON install-window proxy)
2. build a **TRT fp16** engine on the T4 → F1 + inference timing
3. build a **TRT int8** engine (modelopt Q/DQ, MatMul/Gemm-only + fp32) → F1 + timing
4. project the 30k-row DACON runtime → does it fit the 10-min cap?

**Setup:** Runtime ▸ Change runtime type ▸ **T4 GPU**. Then put the artifact bundle in Google Drive at `MyDrive/dacon_trt/`:
- `model_fp16.onnx` (625 MB), `tokenizer.json`, `tokenizer_config.json`, `special_tokens_map.json`
- `val_tokens.npz` (keys: `val_ids`,`val_mask`,`val_y`, `calib_ids`,`calib_mask` — provided alongside this notebook)

In [ ]:
# 0. confirm we are on a T4
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,compute_cap,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert 'T4' in subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout, 'Set runtime to T4 GPU'

In [ ]:
# 1. mount Drive + time the install (DACON allows internet during install <=10min)
import time
from google.colab import drive
drive.mount('/content/drive')
SRC = '/content/drive/MyDrive/dacon_trt'

t0 = time.time()
!pip install -q tensorrt transformers==4.51.3 'nvidia-modelopt[onnx]'
print(f'INSTALL took {time.time()-t0:.0f}s  (DACON install cap = 600s)')

In [ ]:
# 2. load artifacts + shared helpers (torch used only for GPU memory mgmt)
import numpy as np, tensorrt as trt, torch, time, os
from sklearn.metrics import f1_score
SEQ, NCLS, B = 512, 14, 64
logger = trt.Logger(trt.Logger.ERROR)
ONNX = f'{SRC}/model_fp16.onnx'
d = np.load(f'{SRC}/val_tokens.npz')
val_ids, val_mask, val_y = d['val_ids'], d['val_mask'], d['val_y']
calib_ids, calib_mask = d['calib_ids'], d['calib_mask']
print('val rows', len(val_ids), '| calib rows', len(calib_ids))

def build_from_onnx(onnx_path, int8=False):
    b = trt.Builder(logger)
    net = b.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    p = trt.OnnxParser(net, logger)
    assert p.parse(open(onnx_path,'rb').read()), 'parse failed'
    cfg = b.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4<<30)
    cfg.set_flag(trt.BuilderFlag.FP16)
    if int8: cfg.set_flag(trt.BuilderFlag.INT8)
    prof = b.create_optimization_profile()
    for n in ('input_ids','attention_mask'): prof.set_shape(n,(1,SEQ),(B,SEQ),(B,SEQ))
    cfg.add_optimization_profile(prof)
    t0=time.time(); blob=b.build_serialized_network(net,cfg); bs=time.time()-t0
    assert blob is not None, 'engine build failed'
    return trt.Runtime(logger).deserialize_cuda_engine(blob), bs

def infer(engine, ids, mask):
    ctx = engine.create_execution_context()
    d_ids = torch.zeros(B,SEQ,dtype=torch.int64,device='cuda')
    d_mask = torch.zeros(B,SEQ,dtype=torch.int64,device='cuda')
    d_out = torch.zeros(B,NCLS,dtype=torch.float32,device='cuda')
    logits = np.empty((len(ids),NCLS),np.float32)
    torch.cuda.synchronize(); t0=time.time()
    for s in range(0,len(ids),B):
        e=min(s+B,len(ids)); b=e-s
        d_ids[:b]=torch.from_numpy(ids[s:e].astype(np.int64)).cuda()
        d_mask[:b]=torch.from_numpy(mask[s:e].astype(np.int64)).cuda()
        ctx.set_input_shape('input_ids',(b,SEQ)); ctx.set_input_shape('attention_mask',(b,SEQ))
        ctx.set_tensor_address('input_ids',d_ids.data_ptr()); ctx.set_tensor_address('attention_mask',d_mask.data_ptr())
        ctx.set_tensor_address('logits',d_out.data_ptr())
        ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream); torch.cuda.synchronize()
        logits[s:e]=d_out[:b].cpu().numpy()
    return logits, time.time()-t0

In [ ]:
# 3. TRT fp16 on the T4
eng16, build16 = build_from_onnx(ONNX, int8=False)
lo16, inf16 = infer(eng16, val_ids, val_mask)
f1_16 = f1_score(val_y, lo16.argmax(-1), average='macro')
per_row = inf16/len(val_ids)
print(f'fp16: build {build16:.1f}s | infer {inf16:.1f}s ({len(val_ids)} rows) | F1 {f1_16:.4f}')
print(f'  -> projected 30k DACON: build+{build16:.0f}s + infer {per_row*30000:.0f}s = {build16+per_row*30000:.0f}s (cap 600s)')

In [ ]:
# 4. TRT int8 on the T4 (modelopt Q/DQ: MatMul/Gemm-only + fp32 high-precision = the fix)
from modelopt.onnx.quantization import quantize
QDQ = '/content/model_int8.onnx'
t0=time.time()
quantize(onnx_path=ONNX, calibration_data={'input_ids':calib_ids.astype(np.int64),'attention_mask':calib_mask.astype(np.int64)},
         output_path=QDQ, quantize_mode='int8', calibration_method='entropy',
         calibration_eps=['cuda:0'], op_types_to_quantize=['MatMul','Gemm'], high_precision_dtype='fp32')
print(f'modelopt int8 quantize {time.time()-t0:.0f}s -> {os.path.getsize(QDQ)/1e6:.0f}MB')
eng8, build8 = build_from_onnx(QDQ, int8=True)
lo8, inf8 = infer(eng8, val_ids, val_mask)
f1_8 = f1_score(val_y, lo8.argmax(-1), average='macro')
per_row8 = inf8/len(val_ids)
print(f'int8: build {build8:.1f}s | infer {inf8:.1f}s | F1 {f1_8:.4f}  (fp16 F1 {f1_16:.4f})')
print(f'  -> projected 30k DACON: {build8+per_row8*30000:.0f}s (cap 600s) | speedup vs fp16 infer: {inf16/inf8:.2f}x')

In [ ]:
# 5. summary (the numbers that decide shippability on the T4)
print(f'{"engine":<8}{"build_s":>9}{"infer/30k":>11}{"F1":>9}{"fits 10min?":>13}')
for name,bs,pr,f1 in [('fp16',build16,per_row,f1_16),('int8',build8,per_row8,f1_8)]:
    tot=bs+pr*30000
    print(f'{name:<8}{bs:>9.0f}{pr*30000:>11.0f}{f1:>9.4f}{("YES" if tot<600 else "NO"):>13}  (total {tot:.0f}s)')